# Routing experiments: drawing, part photo or text

The agent must decide, without being told, whether a request is a technical drawing (return
every value as JSON), a photo of a part (say what is wrong) or text only. This notebook
measures several ways to make that decision on a labelled set of 244 items, on an M1 Pro with
16 GB and no network (`HF_HUB_OFFLINE=1`).

- **A** deterministic rules: PDF structure, camera EXIF, pixel statistics
- **B** a linear probe (logistic regression) on a frozen image embedding: CLIP, SigLIP 2, DINOv2
- **C** zero-shot CLIP and SigLIP 2 with text prompts, no training
- **D** a local vision-language model asked one routing question (Qwen3-VL 2B, 4B, 8B)
- **F** local open-weight decision models that answer a typed choice with probabilities
  (LiquidAI d1 family through mlx-vlm)
- **G** a pretrained document-figure classifier (Docling DocumentFigureClassifier v2.5)
- **E** the layered router: A, then B when A is undecided, then D or F when B is unsure

The letters are the order the options were written down in, not a ranking. Every measured
result is cached under `results/`, so the notebook re-runs in about a minute without loading
any vision-language model; delete a file there to recompute it.

In [1]:
import gc
import itertools
import json
import os
import subprocess
import sys
import time
import warnings
from collections import Counter
from pathlib import Path

HERE = Path.cwd()  # the notebook runs in its own folder
if not (HERE / "routing_common.py").exists():  # run as a script from elsewhere
    HERE = Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
os.environ["HF_HOME"] = str(HERE / ".cache" / "hf")
os.environ["HF_HUB_OFFLINE"] = "1"  # prove every model runs without the network
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import pymupdf
import routing_common as common
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import LeaveOneGroupOut, StratifiedGroupKFold, StratifiedKFold

try:
    display  # noqa: B018
except NameError:  # running as a plain script
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

ITEMS = common.load_items()
BY_ID = {it["id"]: it for it in ITEMS}
# Items with their own attachment (a message with an attachment reuses another item's file).
ATTACH = [it for it in ITEMS if it.get("file") and it["variant"] != "text_pair"]
MAIN = [it for it in ATTACH if it["label"] in ("drawing", "photo")]
EDGE = [it for it in ATTACH if it["label"] == "other"]  # files that fit no case
PAIRS = [it for it in ITEMS if it["variant"] == "text_pair"]
TEXT = [it for it in ITEMS if it["kind"] == "text"]
print(len(ITEMS), "items:", dict(Counter(it["label"] for it in ITEMS)))
print(
    f"attachments to classify: {len(MAIN)} drawings and photos + {len(EDGE)} that fit no case; "
    f"{len(PAIRS)} messages with an attachment; {len(TEXT)} text only"
)
pd.crosstab([it["label"] for it in ITEMS], [it["variant"] for it in ITEMS])

244 items: {'drawing': 121, 'photo': 93, 'other': 10, 'text': 20}
attachments to classify: 204 drawings and photos + 10 that fit no case; 10 messages with an attachment; 20 text only


col_0,commons,crop,diagram,jpeg,logo,ntsb_figure,ocr_scan_pdf,package_pdf,pdf,photo,photo_pdf,render_100,render_150,render_200,render_300,render_72,report_ocr_scan_pdf,report_page,report_pdf,report_scan_pdf,rotate,scan_pdf,text_only,text_pair
row_0,,,,,,,,,,,,,,,,,,,,,,,,
drawing,7,15,0,15,0,0,1,1,11,15,0,6,6,6,6,6,0,0,0,0,15,6,0,5
other,0,0,1,0,2,0,0,0,0,0,0,0,0,0,0,0,1,2,3,1,0,0,0,0
photo,12,0,0,0,0,71,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,5
text,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,20,0


## Splits

Degraded variants of one page are near duplicates, so a random split leaks: the same sheet
lands on both sides. Every honest split below groups by **source**: one NIST drawing (all its
pages and variants), one NTSB report, or one Commons file. This is the item's `unit`.

Grouping by report page instead is not enough: photos of one specimen on neighbouring pages of
a report then sit on both sides of a fold, which makes the probes look surer than they are.
The page grouping is kept only to show, in section E, how much the escalation rate depends on
the grouping.

In [2]:
main_ids = [it["id"] for it in MAIN]
y_main = np.array([1 if it["label"] == "drawing" else 0 for it in MAIN])  # 1 = drawing
groups = np.array([it["unit"] for it in MAIN])
groups_page = np.array([it["group"] for it in MAIN])  # the finer grouping, by report page
print(len(set(groups)), "source groups,", len(set(groups_page)), "page groups")
HALF = list(  # two grouped halves
    StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=7).split(
        np.zeros(len(MAIN)), y_main, groups
    )
)
RANDOM_HALF = list(  # two random halves, which leak; for comparison only
    StratifiedKFold(n_splits=2, shuffle=True, random_state=7).split(np.zeros(len(MAIN)), y_main)
)
for half, (train, test) in enumerate(HALF):
    print(
        f"grouped half {half}: train {len(train)} ({y_main[train].sum()} drawings), "
        f"test {len(test)} ({y_main[test].sum()} drawings)"
    )


def label_of(p_drawing, threshold=0.5):
    return "drawing" if p_drawing >= threshold else "photo"


def accuracy(y_true, y_pred):
    y_true, y_pred = list(y_true), list(y_pred)
    return sum(a == b for a, b in zip(y_true, y_pred, strict=True)) / len(y_true)

33 source groups, 66 page groups
grouped half 0: train 96 (58 drawings), test 108 (58 drawings)
grouped half 1: train 108 (58 drawings), test 96 (58 drawings)


## A. Deterministic rules

One pass computes every signal (about 40 ms per item, all CPU):

- **PDF**: for each of the first ten pages, the number of vector paths, the characters of real
  text and the share of the page covered by pictures. A page 1 with 300 or more vector paths
  and almost no picture is a drawing. Otherwise the rule counts the first ten pages: a CAD sheet
  has 300 or more paths and little picture, and a page of prose has 1,000 or more characters
  and is not one big picture. A PDF with at least as many CAD sheets as prose pages is a drawing
  package; one with more prose pages is a document (the NTSB reports), which no case in the
  brief covers. A page 1 that is one big picture is judged by the image rules below, so the text
  layer of an OCRed scan never decides. An earlier version of the prose rule used "1,500
  characters on page 1, or 1,000 on any of the first ten pages", which binned a drawing package
  with a notes sheet and an OCRed scan of a report as documents. That earlier rule was itself
  added after the 24-013 report (a cover page with 200 characters) reached the probe, so the
  prose part of layer 1 has been fitted to this set twice.
- **Image**: camera EXIF (Make or Model tag), the share of near-white pixels, and the share of
  "flat" pixels whose Laplacian is near zero (paper and solid fills are flat, photographs are
  textured). Saturation, colour concentration and straight-edge ratio are computed too; they
  did not separate the classes better than these two (see the medians below).

Rules must be high precision: they decide only when sure and say "undecided" otherwise.
The four cut points are fitted on grouped half 0 by grid search: among settings with no error
on that half, take the one that decides the most items. It is then scored on half 1.

In [3]:
# The features are computed and timed once and cached, because a rerun on a busy machine
# measures the load; delete rule_features.json to compute and time them again.
FEATS = common.load_json("rule_features.json")
if FEATS is None:
    FEATS = {}
    for it in ATTACH:
        with common.Timer() as timer:
            FEATS[it["id"]] = common.rule_features(it)
        FEATS[it["id"]]["ms"] = timer.ms
    for it in TEXT:
        FEATS[it["id"]] = {"kind": "text", "ms": 0.0}
    common.save_json("rule_features.json", FEATS)
for it in PAIRS:  # a message with an attachment has its file's features
    FEATS[it["id"]] = FEATS[it["attachment"]]

features_df = pd.DataFrame(
    [
        {"id": it["id"], "label": it["label"], "variant": it["variant"], **FEATS[it["id"]]}
        for it in ATTACH
    ]
)
print(
    f"rule feature time per attachment: median {features_df.ms.median():.0f} ms, "
    f"max {features_df.ms.max():.0f} ms"
)
features_df.groupby(["label", "variant"])[
    [
        "white_frac",
        "flat_frac",
        "sat_mean",
        "top8_share",
        "straight_ratio",
        "pdf_vector_paths",
        "pdf_chars",
    ]
].median().round(2)

rule feature time per attachment: median 38 ms, max 139 ms


white_frac  flat_frac  sat_mean  top8_share  straight_ratio  pdf_vector_paths  pdf_chars
label   variant                                                                                                      
drawing commons                    0.83       0.66      0.00        0.96            0.71               NaN        NaN
        crop                       0.68       0.74      0.09        0.95            0.58               NaN        NaN
        jpeg                       0.79       0.75      0.06        0.93            0.63               NaN        NaN
        ocr_scan_pdf               0.73       0.69      0.00        0.96            0.59               0.0     1284.0
        package_pdf                 NaN        NaN       NaN         NaN             NaN               6.0      144.0
        pdf                         NaN        NaN       NaN         NaN             NaN            1922.0      799.0
        photo                      0.00       0.19      0.27        0.71            0.51               NaN        NaN
        render_100                 0.74       0.74      0.07        0.93            0.61               NaN        NaN
        render_150                 0.78       0.79      0.07        0.94            0.67               NaN        NaN
        render_200                 0.76       0.75      0.06        0.93            0.50               NaN        NaN
        render_300                 0.83       0.81      0.03        0.96            0.63               NaN        NaN
        render_72                  0.79       0.82      0.06        0.96            0.59               NaN        NaN
        rotate                     0.82       0.79      0.05        0.94            0.49               NaN        NaN
        scan_pdf                   0.87       0.80      0.00        0.99            0.56               0.0        0.0
other   diagram                    0.83       0.89      0.12        0.98            0.58               NaN        NaN
        logo                       0.83       0.62      0.12        0.90            0.60               NaN        NaN
        report_ocr_scan_pdf        0.82       0.52      0.00        0.92            0.44               0.0     3512.0
        report_page                0.83       0.65      0.00        0.93            0.51               NaN        NaN
        report_pdf                  NaN        NaN       NaN         NaN             NaN               1.0     2140.0
        report_scan_pdf            0.82       0.52      0.00        0.92            0.44               0.0        0.0
photo   commons                    0.01       0.24      0.11        0.59            0.32               NaN        NaN
        ntsb_figure                0.01       0.20      0.19        0.55            0.36               NaN        NaN
        photo_pdf                  0.01       0.24      0.34        0.46            0.39               0.0        0.0

In [4]:
def rule_decisions(ids, thresholds):
    return [common.decide_rules(FEATS[i], thresholds)[0] for i in ids]


def fit_rules(train_ids, use_exif):
    """The cut points with no wrong decision on train_ids that decide the most of them."""
    truth = [BY_ID[i]["label"] for i in train_ids]
    best_key, best = None, None
    grid = itertools.product(
        [0.3, 0.4, 0.5, 0.6, 0.7],  # drawing_white
        [0.5, 0.6, 0.65, 0.7, 0.75, 0.8],  # drawing_flat
        [0.02, 0.05, 0.1, 0.15, 0.2, 0.3],  # photo_white
        [0.2, 0.3, 0.4, 0.5, 0.6, 0.7],  # photo_flat
    )
    for drawing_white, drawing_flat, photo_white, photo_flat in grid:
        thresholds = common.RuleThresholds(
            use_exif=use_exif,
            drawing_white=drawing_white,
            drawing_flat=drawing_flat,
            photo_white=photo_white,
            photo_flat=photo_flat,
        )
        decided = rule_decisions(train_ids, thresholds)
        pairs = list(zip(decided, truth, strict=True))
        wrong = sum(d in ("drawing", "photo") and d != y for d, y in pairs)
        covered = sum(d in ("drawing", "photo") for d in decided)
        key = (-wrong, covered)
        if best_key is None or key > best_key:
            best_key, best = key, thresholds
    return best


def score_rules(ids, thresholds):
    truth = [BY_ID[i]["label"] for i in ids]
    decisions = rule_decisions(ids, thresholds)
    decided = [(d, y) for d, y in zip(decisions, truth, strict=True) if d != "undecided"]
    return {
        "items": len(ids),
        "coverage": len(decided) / len(ids),
        "precision_when_decided": accuracy([y for _, y in decided], [d for d, _ in decided])
        if decided
        else None,
        "accuracy_undecided_counts_wrong": accuracy(truth, decisions),
    }


RULE_FITS = {}
train_ids = [main_ids[i] for i in HALF[0][0]]
test_ids = [main_ids[i] for i in HALF[0][1]]
for use_exif in (True, False):
    thresholds = fit_rules(train_ids, use_exif)
    RULE_FITS[f"exif={use_exif}"] = {
        "thresholds": thresholds.__dict__,
        "train_half": score_rules(train_ids, thresholds),
        "test_half": score_rules(test_ids, thresholds),
        "all_main": score_rules(main_ids, thresholds),
    }
RULES_EXIF = common.RuleThresholds(**RULE_FITS["exif=True"]["thresholds"])
RULES_NO_EXIF = common.RuleThresholds(**RULE_FITS["exif=False"]["thresholds"])
pd.DataFrame(
    {
        name: {
            **fit["thresholds"],
            **{f"test {k}": v for k, v in fit["test_half"].items()},
            **{f"all {k}": v for k, v in fit["all_main"].items()},
        }
        for name, fit in RULE_FITS.items()
    }
)

,exif=True,exif=False
use_exif,True,False
drawing_white,0.3,0.3
drawing_flat,0.7,0.8
photo_white,0.02,0.02
photo_flat,0.2,0.2
pdf_vector_paths,300,300
pdf_prose_page_chars,1000,1000
full_page_picture,0.9,0.9
test items,108,108
test coverage,0.592593,0.37963


In [5]:
def rules_table(thresholds):
    """Every item with the rules' decision and its reason."""
    rows = []
    for it in ITEMS:
        decision, why = common.decide_rules(FEATS[it["id"]], thresholds)
        rows.append(
            {
                "id": it["id"],
                "label": it["label"],
                "variant": it["variant"],
                "decision": decision,
                "why": why,
            }
        )
    return pd.DataFrame(rows)


rules_exif = rules_table(RULES_EXIF)
print("Approach A with EXIF, all items (text included):")
display(pd.crosstab(rules_exif.label, rules_exif.decision, margins=True))
print("Where A decided wrongly:")
display(
    rules_exif[
        rules_exif.decision.isin(["drawing", "photo", "text"])
        & (rules_exif.decision != rules_exif.label)
        & (rules_exif.label != "other")
    ]
)
print("What A left undecided, by variant:")
display(
    rules_exif[rules_exif.decision == "undecided"]
    .groupby(["label", "variant"])
    .size()
    .rename("undecided")
)
rules_no_exif = rules_table(RULES_NO_EXIF)
print("Without the EXIF rule:")
display(pd.crosstab(rules_no_exif.label, rules_no_exif.decision, margins=True))

Approach A with EXIF, all items (text included):


decision,document,drawing,photo,text,undecided,All
label,,,,,,
drawing,0,93,15,0,13,121
other,3,2,0,0,5,10
photo,0,0,30,0,63,93
text,0,0,0,20,0,20
All,3,95,45,20,81,244


Where A decided wrongly:


,id,label,variant,decision,why
3,d_ctc01_p1_photo,drawing,photo,photo,camera EXIF
13,d_ctc02_p3_photo,drawing,photo,photo,"white 0.00, flat 0.18"
20,d_ctc04_p1_photo,drawing,photo,photo,camera EXIF
25,d_ctc05_p1_photo,drawing,photo,photo,"white 0.02, flat 0.20"
35,d_ftc06_p2_photo,drawing,photo,photo,camera EXIF
41,d_ftc07_p1_photo,drawing,photo,photo,"white 0.01, flat 0.19"
45,d_ftc07_p2_photo,drawing,photo,photo,camera EXIF
55,d_ftc08_p1_photo,drawing,photo,photo,"white 0.02, flat 0.18"
60,d_ftc08_p3_photo,drawing,photo,photo,camera EXIF
64,d_ftc08_p4_photo,drawing,photo,photo,"white 0.00, flat 0.19"


What A left undecided, by variant:


label    variant            
drawing  commons                 4
         crop                    2
         jpeg                    3
         ocr_scan_pdf            1
         photo                   1
         render_100              1
         rotate                  1
other    logo                    1
         report_ocr_scan_pdf     1
         report_page             2
         report_scan_pdf         1
photo    commons                 1
         ntsb_figure            54
         photo_pdf               4
         text_pair               4
Name: undecided, dtype: int64

Without the EXIF rule:


decision,document,drawing,photo,text,undecided,All
label,,,,,,
drawing,0,40,14,0,67,121
other,3,1,0,0,6,10
photo,0,0,22,0,71,93
text,0,0,0,20,0,20
All,3,41,36,20,144,244


**Structure-only rules.** Every wrong decision above is a drawing photographed on a desk: its
pixels look like a photo, and 8 of the 15 phone copies carry phone EXIF. So the variant used
as layer 1 of the layered router keeps only the signals that come from the file's structure:
no attachment means text, a vector PDF or drawing package means drawing, a prose PDF means
document. Every raster image, and every PDF that is one big picture, goes on to layer 2.

In [6]:
# Image cut points that can never be met, so only the PDF and text rules decide.
RULES_STRUCTURE = common.RuleThresholds(
    use_exif=False, drawing_white=2.0, drawing_flat=2.0, photo_white=-1.0, photo_flat=-1.0
)
pdf_items = [it for it in ATTACH if it["kind"] == "pdf"]
saved_rules = common.load_json("A_rules.json") or {}

# The structure check at scale: PyMuPDF's get_cdrawings() returns the same paths as
# get_drawings() as plain tuples, which its documentation says is much faster.
# Like every timing here, it is measured once and cached; delete A_rules.json to time it again.
STRUCT_MS = saved_rules.get("struct_check_ms")
if STRUCT_MS is None:
    page1_ms = {"get_drawings": [], "get_cdrawings": []}
    for it in pdf_items:
        for method in page1_ms:
            start = time.perf_counter()
            with pymupdf.open(common.item_path(it)) as doc:
                page = doc[0]
                getattr(page, method)()
                page.get_text()
                page.get_images(full=True)
            page1_ms[method].append((time.perf_counter() - start) * 1000)
    STRUCT_MS = {
        method: {"median": float(np.median(ms)), "max": float(np.max(ms))}
        for method, ms in page1_ms.items()
    }
print("page 1 structure check per PDF, ms:", STRUCT_MS)


def layer1_check(path):
    """The layer-1 check as a router would run it: page 1 first, and pages 2 to 10 only when
    page 1 is not a CAD sheet."""
    with pymupdf.open(path) as doc:
        paths, _, cover = common.page_layout(doc[0])
        if paths >= RULES_STRUCTURE.pdf_vector_paths and cover < 0.2:
            return
        for i in range(1, min(10, doc.page_count)):
            common.page_layout(doc[i])


# Timed per PDF as the median of three runs, and cached; section E uses these times.
STRUCT10_MS = saved_rules.get("struct_check_layer1_ms", {})
for it in pdf_items:
    if it["id"] not in STRUCT10_MS:
        runs = []
        for _ in range(3):
            start = time.perf_counter()
            layer1_check(common.item_path(it))
            runs.append((time.perf_counter() - start) * 1000)
        STRUCT10_MS[it["id"]] = float(np.median(runs))
print(
    f"layer-1 check per PDF: median {np.median(list(STRUCT10_MS.values())):.1f} ms, "
    f"max {max(STRUCT10_MS.values()):.1f} ms"
)
display(
    pd.DataFrame(
        {"ms": [STRUCT10_MS[it["id"]] for it in pdf_items]},
        index=[it["variant"] for it in pdf_items],
    )
    .groupby(level=0)
    .median()
    .round(1)
)
rules_structure = rules_table(RULES_STRUCTURE)


def earlier_prose_rule(f):
    """The earlier prose rule, kept only to show what counting page kinds changed."""
    if f["pdf_vector_paths"] >= 300 and f["pdf_image_cover"] < 0.2:
        return "drawing"
    if f["pdf_chars"] >= 1500 and f["pdf_vector_paths"] < 300:
        return "document"
    if f["pdf_image_cover"] < 0.5 and f["pdf_max_chars_first10"] >= 1000:
        return "document"
    return "undecided"


now = dict(zip(rules_structure.id, rules_structure.decision, strict=True))
pdf_calls = pd.DataFrame(
    [
        {
            "id": it["id"],
            "label": it["label"],
            "earlier rule": earlier_prose_rule(FEATS[it["id"]]),
            "now": now[it["id"]],
        }
        for it in pdf_items
    ]
)
print("PDFs whose layer-1 call changed when the rule started counting page kinds:")
display(pdf_calls[pdf_calls["earlier rule"] != pdf_calls["now"]])
display(pd.crosstab(rules_structure.label, rules_structure.decision, margins=True))
common.save_json(
    "A_rules.json",
    {
        "fits": RULE_FITS,
        "decisions": rules_exif.to_dict("records"),
        "decisions_noexif": rules_no_exif.to_dict("records"),
        "decisions_structural": rules_structure.to_dict("records"),
        "feature_ms_median": float(features_df.ms.median()),
        "feature_ms_pdf_median": float(features_df[features_df.kind == "pdf"].ms.median()),
        "feature_ms_image_median": float(features_df[features_df.kind == "image"].ms.median()),
        "struct_check_ms": STRUCT_MS,
        "struct_check_layer1_ms": STRUCT10_MS,
        "pdf_calls": pdf_calls.to_dict("records"),
    },
)

page 1 structure check per PDF, ms: {'get_drawings': {'median': 5.440125009045005, 'max': 99.01362494565547}, 'get_cdrawings': {'median': 3.464166074991226, 'max': 30.09462496265769}}
layer-1 check per PDF: median 6.9 ms, max 50.5 ms


,ms
ocr_scan_pdf,3.5
package_pdf,50.5
pdf,21.2
photo_pdf,1.1
report_ocr_scan_pdf,4.5
report_pdf,30.3
report_scan_pdf,1.4
scan_pdf,1.1


PDFs whose layer-1 call changed when the rule started counting page kinds:


,id,label,earlier rule,now
25,d_ftc06_package_pdf,drawing,document,drawing
28,o_report_18061_page5_ocrscan,other,document,undecided


decision,document,drawing,text,undecided,All
label,,,,,
drawing,0,14,0,107,121
other,3,0,0,7,10
photo,0,0,0,93,93
text,0,0,20,0,20
All,3,14,20,207,244


**What the structure rule counts.** A PDF page draws lines and shapes as vector paths. Each
path is a list of pieces (straight lines `l`, curves `c`, rectangles `re`, four-sided shapes
`qu`) that is then stroked (`s`), filled (`f`) or both (`fs`). A CAD export writes every
outline, dimension line, arrowhead and shaded face as paths, while a scan or a photo saved as a
PDF is one embedded picture with no paths. `get_drawings()` returns one entry per path on the
page. The table shows page 1 of every PDF in the set.

In [7]:
pdf_structure = []
for it in pdf_items:
    with pymupdf.open(common.item_path(it)) as doc:
        paths = doc[0].get_drawings()
    pdf_structure.append(
        {
            "id": it["id"],
            "label": it["label"],
            "variant": it["variant"],
            "paths": len(paths),
            "path_types": dict(Counter(path["type"] for path in paths)),
            "pieces": dict(Counter(piece[0] for path in paths for piece in path["items"])),
            "image_cover": FEATS[it["id"]]["pdf_image_cover"],
            "chars": FEATS[it["id"]]["pdf_chars"],
        }
    )
pdf_structure_summary = (
    pd.DataFrame(pdf_structure)
    .groupby("variant")
    .agg(
        files=("id", "size"),
        paths_min=("paths", "min"),
        paths_median=("paths", "median"),
        paths_max=("paths", "max"),
        image_cover_median=("image_cover", "median"),
        chars_median=("chars", "median"),
    )
    .reset_index()
)
display(pdf_structure_summary.round(2))
common.save_json(
    "A_pdf_structure.json",
    {"files": pdf_structure, "by_variant": pdf_structure_summary.to_dict("records")},
)

,variant,files,paths_min,paths_median,paths_max,image_cover_median,chars_median
0,ocr_scan_pdf,1,0,0.0,0,1.00,1284.0
1,package_pdf,1,6,6.0,6,0.00,144.0
2,pdf,11,613,1922.0,5319,0.00,799.0
3,photo_pdf,5,0,0.0,0,1.00,0.0
4,report_ocr_scan_pdf,1,0,0.0,0,1.00,3512.0
5,report_pdf,3,1,1.0,17,0.02,2140.0
6,report_scan_pdf,1,0,0.0,0,1.00,0.0
7,scan_pdf,6,0,0.0,0,1.00,0.0


## Embeddings for B and C

Three frozen image encoders, run with PyTorch on the M1 GPU (MPS), batch size 1 so the
latency is what one upload would see. The input is the attachment image or page 1 of a PDF
rendered to 1,024 px on its long side; each model's own processor then resizes it to 224 px.

- CLIP ViT-B/32 (OpenAI weights through OpenCLIP, 151M parameters, MIT licence)
- SigLIP 2 base patch16 224 (Google, 375M parameters, Apache-2.0)
- DINOv2 small (Meta, 22M parameters, Apache-2.0); image only, so no zero-shot

In [8]:
import torch

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"


def load_encoder(name):
    """(embed an image, embed texts, logit scale, logit bias) for one image model."""
    if name == "clip_b32":
        import open_clip

        # pretrained="openai" resolves to timm/vit_base_patch32_clip_224.openai on the HF cache
        model, _, preprocess = open_clip.create_model_and_transforms(
            "ViT-B-32", pretrained="openai"
        )
        tokenizer = open_clip.get_tokenizer("ViT-B-32")
        model = model.to(DEVICE).eval()

        def embed_image(img):
            return model.encode_image(preprocess(img).unsqueeze(0).to(DEVICE))

        def embed_texts(texts):
            return model.encode_text(tokenizer(texts).to(DEVICE))

        return embed_image, embed_texts, float(model.logit_scale.exp()), None
    if name == "siglip2_b16":
        from transformers import AutoModel, AutoProcessor

        repo = "google/siglip2-base-patch16-224"
        model = AutoModel.from_pretrained(repo).to(DEVICE).eval()
        processor = AutoProcessor.from_pretrained(repo)

        def embed_image(img):
            out = model.get_image_features(**processor(images=img, return_tensors="pt").to(DEVICE))
            return getattr(out, "pooler_output", out)

        def embed_texts(texts):
            inputs = processor(
                text=texts, padding="max_length", max_length=64, return_tensors="pt"
            ).to(DEVICE)
            out = model.get_text_features(**inputs)
            return getattr(out, "pooler_output", out)

        return embed_image, embed_texts, float(model.logit_scale.exp()), float(model.logit_bias)
    if name == "dinov2_s":
        from transformers import AutoImageProcessor, AutoModel

        repo = "facebook/dinov2-small"
        model = AutoModel.from_pretrained(repo).to(DEVICE).eval()
        processor = AutoImageProcessor.from_pretrained(repo)

        def embed_image(img):
            return model(**processor(images=img, return_tensors="pt").to(DEVICE)).pooler_output

        return embed_image, None, None, None
    raise ValueError(name)


def unit_vector(tensor):
    return torch.nn.functional.normalize(tensor.float(), dim=-1).cpu().numpy()


def free_gpu_memory():
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()


ENCODERS = ["clip_b32", "siglip2_b16", "dinov2_s"]
EMB, EMB_STATS, TXT = {}, {}, {}
ZS_PROMPTS = {
    "drawing": [
        "a technical drawing",
        "an engineering drawing with dimensions and tolerances",
        "a CAD drawing sheet with a title block",
        "a photo of a printed engineering drawing",
    ],
    "photo": [
        "a photograph of a metal part",
        "a close-up photo of a fractured metal surface",
        "a microscope image of a metal surface",
        "a photo of a broken mechanical part",
    ],
}


def embed_all(name):
    """Embed every attachment and the zero-shot prompts, timing each image after a warm-up."""
    rss_before = common.rss_mb()
    start = time.perf_counter()
    embed_image, embed_texts, scale, bias = load_encoder(name)
    load_s = time.perf_counter() - start
    vectors, ms = {}, []
    with torch.no_grad():
        for k, it in enumerate(ATTACH):
            with common.Timer() as timer:
                vectors[it["id"]] = unit_vector(embed_image(common.attachment_image(it, 1024)))[0]
            if k >= 3:  # the first three are warm-up
                ms.append(timer.ms)
        text = None
        if embed_texts is not None:
            prompts = {c: unit_vector(embed_texts(ps)).tolist() for c, ps in ZS_PROMPTS.items()}
            text = {"emb": prompts, "scale": scale, "bias": bias}
    stats = {
        "load_s": load_s,
        "ms_median": float(np.median(ms)),
        "ms_p95": float(np.percentile(ms, 95)),
        "rss_delta_mb": common.rss_mb() - rss_before,
        "mps_driver_mb": torch.mps.driver_allocated_memory() / 1e6 if DEVICE == "mps" else None,
        "dim": int(len(next(iter(vectors.values())))),
    }
    del embed_image, embed_texts
    free_gpu_memory()
    return vectors, stats, text


for name in ENCODERS:
    path = common.RESULTS / f"emb_{name}.npz"
    if not path.exists():
        vectors, stats, text = embed_all(name)
        np.savez(
            path,
            ids=np.array(list(vectors)),
            emb=np.stack(list(vectors.values())),
            stats=json.dumps(stats),
            **({"txt": json.dumps(text)} if text else {}),
        )
    saved = np.load(path, allow_pickle=True)
    EMB[name] = dict(zip(saved["ids"], saved["emb"], strict=True))
    EMB_STATS[name] = json.loads(str(saved["stats"]))
    if "txt" in saved:
        TXT[name] = json.loads(str(saved["txt"]))
pd.DataFrame(EMB_STATS).T.round(1)

,load_s,ms_median,ms_p95,rss_delta_mb,mps_driver_mb,dim
clip_b32,4.3,30.6,67.4,747.5,1615.5,512.0
siglip2_b16,2.3,39.5,77.9,-169.1,1548.4,768.0
dinov2_s,0.2,32.5,72.4,57.3,113.9,384.0


## B. Linear probe on the embedding

Logistic regression (scikit-learn, L2, balanced class weights) on the normalised embedding;
class 1 is drawing. The image model stays frozen: only this regression is trained.

The regularisation strength C is fixed at 100 for every probe. Over a grid from 0.1 to 100,000
the grouped log loss keeps falling as C grows and flattens from about 1,000, because the
drawings and photos here are easy to separate; from 100 to 10,000 it improves by about 0.01.
A larger C also makes the probe sure of files that are neither a drawing nor a photo, so fewer
of them escalate (the curve below). C stays at 100 for that reason, which means the choice
looked at the files that fit no case; it should be made again on a customer's own files, with
real examples of files that fit no case. Six checks:

1. **random half**: train on half the items, test on the other half, ignoring groups (leaky)
2. **grouped half**: the same, but a source sits wholly on one side; both directions are
   reported
3. **leave one drawing out**: train without any page or variant of one NIST drawing, test on
   that drawing; repeated for all 11
4. **leave one report out**: the same for the NTSB photo sources
5. **out of domain**: train on NIST and NTSB only, test on the Wikimedia Commons photos and
   drawings, which come from other cameras, eras and drawing styles
6. **reverse family**: train on the 7 Commons drawings and every photo, test on the whole NIST
   family; this is what a customer whose drawings look like no training drawing would see

The out-of-fold probabilities the layered router uses come from 5 folds grouped by source.
They are also computed with folds grouped by report page and with one source left out at a
time, because the escalation rate depends on the grouping.

In [9]:
def embeddings(name, ids):
    return np.stack([EMB[name][i] for i in ids])


C_FIXED = 100.0


def new_probe():
    return LogisticRegression(C=C_FIXED, max_iter=5000, class_weight="balanced")


def c_curve(name):
    """Grouped 5-fold log loss, and how many files that fit no case stay below 0.9, for each C."""
    X, y = embeddings(name, main_ids), y_main
    X_edge = embeddings(name, [it["id"] for it in EDGE])
    curve = {}
    for c in (0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0):
        oof = np.zeros(len(y))
        folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=3)
        for train, test in folds.split(X, y, groups):
            model = LogisticRegression(C=c, max_iter=5000, class_weight="balanced")
            oof[test] = model.fit(X[train], y[train]).predict_proba(X[test])[:, 1]
        model = LogisticRegression(C=c, max_iter=5000, class_weight="balanced").fit(X, y)
        p_edge = model.predict_proba(X_edge)[:, 1]
        curve[c] = {
            "log_loss": float(log_loss(y, oof)),
            "edge_below_0_9": int((np.maximum(p_edge, 1 - p_edge) < 0.9).sum()),
        }
    return curve


def held_out_accuracy(X, y, test):
    """Accuracy on the files in test of a probe trained on all the others."""
    pred = new_probe().fit(X[~test], y[~test]).predict(X[test])
    return float((pred == y[test]).mean()), pred


def expected_calibration_error(p_drawing, y):
    """The gap between confidence and accuracy, averaged over five confidence bins."""
    confidence = np.maximum(p_drawing, 1 - p_drawing)
    bins = np.linspace(0.5, 1.0, 6)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:], strict=True):
        in_bin = (confidence >= lo) & (confidence < hi if hi < 1 else confidence <= hi)
        if in_bin.any():
            right = ((p_drawing[in_bin] >= 0.5) == y[in_bin]).mean()
            ece += in_bin.mean() * abs(right - confidence[in_bin].mean())
    return float(ece)


def out_of_fold(X, y, split_groups, splitter):
    """Each file's P(drawing) from a probe trained on the folds that do not hold it."""
    oof = np.zeros(len(y))
    for train, test in splitter.split(X, y, split_groups):
        oof[test] = new_probe().fit(X[train], y[train]).predict_proba(X[test])[:, 1]
    return oof


def run_probe(name):
    X, y = embeddings(name, main_ids), y_main
    sources = np.array([it["source"] for it in MAIN])
    out = {"C": C_FIXED, "c_curve": c_curve(name)}
    for tag, splits in (("random_half", RANDOM_HALF), ("grouped_half", HALF)):
        out[tag] = [
            float((new_probe().fit(X[train], y[train]).predict(X[test]) == y[test]).mean())
            for train, test in splits
        ]

    by_drawing, wrong = {}, []
    for drawing in sorted({it["group"] for it in MAIN if it["source"] == "nist"}):
        test = groups == drawing
        by_drawing[drawing], pred = held_out_accuracy(X, y, test)
        wrong += [main_ids[i] for i in np.where(test)[0][pred != y[test]]]
    out["lodo_mean"] = float(np.mean(list(by_drawing.values())))
    out["lodo_min"] = float(min(by_drawing.values()))
    out["lodo"] = by_drawing
    out["lodo_wrong"] = wrong

    out["loro"] = {
        report: held_out_accuracy(X, y, sources == report)[0]
        for report in sorted({s for s in sources if s.startswith("ntsb")})
    }

    test = sources == "commons"
    out["ood_commons"], pred = held_out_accuracy(X, y, test)
    out["ood_wrong"] = [main_ids[i] for i in np.where(test)[0][pred != y[test]]]

    # reverse family: train on the Commons drawings and every photo, score the NIST family
    test = sources == "nist"
    p_nist = new_probe().fit(X[~test], y[~test]).predict_proba(X[test])[:, 1]
    wrong_nist = (p_nist >= 0.5) != y[test]
    out["reverse_nist"] = {
        "n": int(test.sum()),
        "wrong": int(wrong_nist.sum()),
        "below_0_9": int((np.maximum(p_nist, 1 - p_nist) < 0.9).sum()),
        "wrong_ids": [main_ids[i] for i in np.where(test)[0][wrong_nist]],
    }

    # out-of-fold probabilities for the layered router, under three groupings
    five_folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=11)
    oof = out_of_fold(X, y, groups, five_folds)
    out["oof_by_grouping"] = {
        "page": dict(
            zip(main_ids, out_of_fold(X, y, groups_page, five_folds).tolist(), strict=True)
        ),
        "loso": dict(
            zip(main_ids, out_of_fold(X, y, groups, LeaveOneGroupOut()).tolist(), strict=True)
        ),
    }
    out["oof_acc"] = float(((oof >= 0.5) == y).mean())
    out["oof_wrong"] = [main_ids[i] for i in np.where((oof >= 0.5) != y)[0]]
    out["oof_p_drawing"] = dict(zip(main_ids, oof.tolist(), strict=True))
    out["brier"] = float(np.mean((oof - y) ** 2))
    out["ece"] = expected_calibration_error(oof, y)

    # the files that fit no case, scored by a probe trained on every drawing and photo (the
    # weights a router would ship), and that probe's lowest confidence on its own training files
    full = new_probe().fit(X, y)
    p_edge = full.predict_proba(embeddings(name, [it["id"] for it in EDGE]))[:, 1]
    out["edge"] = {it["id"]: float(p) for it, p in zip(EDGE, p_edge, strict=True)}
    p_train = full.predict_proba(X)[:, 1]
    out["in_sample_min_conf"] = float(np.maximum(p_train, 1 - p_train).min())

    # training and inference time, measured once and cached like the other timings
    if "fit_ms" in PROBE_TIMES.get(name, {}):
        out["fit_ms"] = PROBE_TIMES[name]["fit_ms"]
        out["predict_ms"] = PROBE_TIMES[name]["predict_ms"]
        return out
    start = time.perf_counter()
    full = new_probe().fit(X, y)
    out["fit_ms"] = (time.perf_counter() - start) * 1000
    start = time.perf_counter()
    for _ in range(200):
        full.predict_proba(X[:1])
    out["predict_ms"] = (time.perf_counter() - start) * 1000 / 200
    return out


PROBE_TIMES = common.load_json("B_probe.json") or {}
PROBES = {name: run_probe(name) for name in ENCODERS}
common.save_json("B_probe.json", PROBES)
print("log loss and edge files below 0.9 by C:")
display(
    pd.DataFrame(
        {
            (name, metric): {c: v[metric] for c, v in probe["c_curve"].items()}
            for name, probe in PROBES.items()
            for metric in ("log_loss", "edge_below_0_9")
        }
    ).round(4)
)
pd.DataFrame(
    {
        name: {
            "C": probe["C"],
            "random half (leaky)": np.mean(probe["random_half"]),
            "grouped half 0->1": probe["grouped_half"][0],
            "grouped half 1->0": probe["grouped_half"][1],
            "leave one drawing out, mean": probe["lodo_mean"],
            "leave one drawing out, worst": probe["lodo_min"],
            **{f"leave out {k}": v for k, v in probe["loro"].items()},
            "out of domain (Commons)": probe["ood_commons"],
            "reverse: NIST wrong": probe["reverse_nist"]["wrong"],
            "reverse: NIST below 0.9": probe["reverse_nist"]["below_0_9"],
            "5-fold grouped OOF": probe["oof_acc"],
            "Brier": probe["brier"],
            "ECE": probe["ece"],
            "embed ms (MPS)": EMB_STATS[name]["ms_median"],
            "probe ms": probe["predict_ms"],
        }
        for name, probe in PROBES.items()
    }
).round(3)

log loss and edge files below 0.9 by C:


clip_b32                siglip2_b16                dinov2_s               
         log_loss edge_below_0_9    log_loss edge_below_0_9 log_loss edge_below_0_9
0.1        0.5067             10      0.5017             10   0.4531             10
1.0        0.2253             10      0.2172             10   0.1776             10
10.0       0.0772             10      0.0713             10   0.0573              7
100.0      0.0264              9      0.0237             10   0.0218              5
1000.0     0.0124              7      0.0100              6   0.0128              4
10000.0    0.0116              7      0.0053              2   0.0116              2
100000.0   0.0114              7      0.0070              2   0.0118              2

,clip_b32,siglip2_b16,dinov2_s
C,100.000,100.000,100.000
random half (leaky),1.000,1.000,1.000
grouped half 0->1,1.000,1.000,1.000
grouped half 1->0,1.000,1.000,0.990
"leave one drawing out, mean",1.000,1.000,1.000
"leave one drawing out, worst",1.000,1.000,1.000
leave out ntsb_16-006,1.000,1.000,1.000
leave out ntsb_18-061,1.000,1.000,1.000
leave out ntsb_24-013,1.000,1.000,1.000
out of domain (Commons),0.895,0.895,1.000


In [10]:
for name, probe in PROBES.items():
    print(
        name,
        "| OOF wrong:",
        probe["oof_wrong"],
        "| OOD wrong:",
        probe["ood_wrong"],
        "| LODO wrong:",
        probe["lodo_wrong"],
    )
    print("   edge p(drawing):", {k: round(v, 2) for k, v in probe["edge"].items()})

clip_b32 | OOF wrong: [] | OOD wrong: ['d_commons_albertkoppelritning', 'd_commons_werkstattzeichnungdesnor'] | LODO wrong: []
   edge p(drawing): {'o_logo_18061_x677': 0.86, 'o_logo_24013_x159': 0.45, 'o_report_16006_pdf': 0.69, 'o_report_18061_pdf': 0.68, 'o_report_24013_pdf': 0.82, 'o_report_16006_page2': 0.26, 'o_report_18061_page3': 0.29, 'o_commons_abmessungen': 0.94, 'o_report_18061_page5_scan': 0.41, 'o_report_18061_page5_ocrscan': 0.41}
siglip2_b16 | OOF wrong: [] | OOD wrong: ['d_commons_albertkoppelritning', 'd_commons_werkstattzeichnungdesnor'] | LODO wrong: []
   edge p(drawing): {'o_logo_18061_x677': 0.26, 'o_logo_24013_x159': 0.77, 'o_report_16006_pdf': 0.72, 'o_report_18061_pdf': 0.68, 'o_report_24013_pdf': 0.84, 'o_report_16006_page2': 0.76, 'o_report_18061_page3': 0.69, 'o_commons_abmessungen': 0.86, 'o_report_18061_page5_scan': 0.74, 'o_report_18061_page5_ocrscan': 0.74}
dinov2_s | OOF wrong: [] | OOD wrong: [] | LODO wrong: []
   edge p(drawing): {'o_logo_18061_x677

## C. Zero-shot with text prompts

No training. Each class is described by four prompts; the image is assigned to the class
whose mean prompt embedding is closest. CLIP turns the similarities into a softmax with its
learned temperature; SigLIP 2 scores each prompt with its own sigmoid and we compare the class
means. The prompts include "a photo of a printed engineering drawing" because a phone photo
of a sheet must still route to the drawing case.

In [11]:
def zero_shot(name):
    """Each attachment's P(drawing) from its similarity to the prompts of each class."""
    text = TXT[name]
    prompts = {c: np.array(text["emb"][c]) for c in ZS_PROMPTS}
    out = {}
    for it in ATTACH:
        vector = EMB[name][it["id"]]
        if name.startswith("siglip"):
            sigmoid = {
                c: float(
                    np.mean(1 / (1 + np.exp(-(text["scale"] * prompts[c] @ vector + text["bias"]))))
                )
                for c in prompts
            }
            total = sum(sigmoid.values())
            out[it["id"]] = sigmoid["drawing"] / total if total > 0 else 0.5
        else:
            logits = np.array([np.mean(text["scale"] * prompts[c] @ vector) for c in prompts])
            softmax = np.exp(logits - logits.max())
            out[it["id"]] = float(softmax[0] / softmax.sum())
    return out


ZERO_SHOT = {}
truth_main = [it["label"] for it in MAIN]
for name in ("clip_b32", "siglip2_b16"):
    p_drawing = zero_shot(name)
    pred = [label_of(p_drawing[i]) for i in main_ids]
    ZERO_SHOT[name] = {
        "acc": accuracy(truth_main, pred),
        "p_drawing": p_drawing,
        "wrong": [i for i, a, b in zip(main_ids, truth_main, pred, strict=True) if a != b],
        "confusion": common.confusion(truth_main, pred, rows=("drawing", "photo")),
    }
common.save_json("C_zeroshot.json", ZERO_SHOT)
for name, result in ZERO_SHOT.items():
    print(f"{name}: accuracy {result['acc']:.3f}; confusion {result['confusion']}")
    print("   wrong:", result["wrong"])

clip_b32: accuracy 0.990; confusion {'drawing': {'drawing': 116, 'photo': 0}, 'photo': {'drawing': 2, 'photo': 86}}
   wrong: ['p_16006_p08_x63', 'p_16006_p09_x74']
siglip2_b16: accuracy 1.000; confusion {'drawing': {'drawing': 116, 'photo': 0}, 'photo': {'drawing': 0, 'photo': 88}}
   wrong: []


## G. A pretrained document-figure classifier (Docling)

Docling's DocumentFigureClassifier v2.5 is an EfficientNet-B0 (4M parameters, 34 MB, MIT
licence) trained to sort figures cut out of PDFs into 26 classes, two of which are
`engineering_drawing` and `photograph`. It needs no training from us. We map
`engineering_drawing` to drawing, `photograph` to photo, and the other 24 classes to other,
and take the group with the highest summed probability. Its 1,280-number pooled features are
also tried as a fourth embedding for the linear probe.

In [12]:
docling_path = common.RESULTS / "G_docling.json"
docling_features_path = common.RESULTS / "emb_docling_fig.npz"
if not docling_path.exists():
    import torchvision.transforms as T
    from transformers import EfficientNetForImageClassification

    repo = "docling-project/DocumentFigureClassifier-v2.5"
    figure_model = EfficientNetForImageClassification.from_pretrained(repo).to(DEVICE).eval()
    to_input = T.Compose(
        [
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.47853944, 0.4732864, 0.47434163]),
        ]
    )
    class_names = figure_model.config.id2label
    docling_items, features, ms = {}, {}, []
    with torch.no_grad():
        for k, it in enumerate(ATTACH):
            with common.Timer() as timer:
                x = to_input(common.attachment_image(it, 1024)).unsqueeze(0).to(DEVICE)
                hidden = figure_model.efficientnet(x).pooler_output
                probs = figure_model.classifier(hidden).softmax(-1).float().cpu().numpy()[0]
            if k >= 3:  # the first three are warm-up
                ms.append(timer.ms)
            ranked = sorted(
                ((class_names[i], float(p)) for i, p in enumerate(probs)), key=lambda pair: -pair[1]
            )
            docling_items[it["id"]] = {
                "probs": {class_names[i]: float(p) for i, p in enumerate(probs)},
                "top": ranked[:3],
            }
            features[it["id"]] = unit_vector(hidden)[0]
    docling_stats = {
        "ms_median": float(np.median(ms)),
        "ms_p95": float(np.percentile(ms, 95)),
        "mps_driver_mb": torch.mps.driver_allocated_memory() / 1e6 if DEVICE == "mps" else None,
    }
    docling_path.write_text(json.dumps({"items": docling_items, "stats": docling_stats}))
    np.savez(
        docling_features_path,
        ids=np.array(list(features)),
        emb=np.stack(list(features.values())),
        stats=json.dumps({**docling_stats, "dim": 1280}),
    )
    del figure_model
    free_gpu_memory()
DOCLING_RAW = json.loads(docling_path.read_text())


def docling_route(probs):
    """drawing, photo or other, whichever has the largest summed probability."""
    drawing = probs["engineering_drawing"]
    photo = probs["photograph"]
    return max(
        (("drawing", drawing), ("photo", photo), ("other", 1 - drawing - photo)),
        key=lambda pair: pair[1],
    )[0]


def docling_two_way(probs):
    return "drawing" if probs["engineering_drawing"] >= probs["photograph"] else "photo"


docling_probs = {i: DOCLING_RAW["items"][i]["probs"] for i in DOCLING_RAW["items"]}
docling_pred = [docling_route(docling_probs[i]) for i in main_ids]
DOCLING = {
    "acc_3way": accuracy(truth_main, docling_pred),
    "acc_2way": accuracy(truth_main, [docling_two_way(docling_probs[i]) for i in main_ids]),
    "confusion": common.confusion(truth_main, docling_pred, rows=("drawing", "photo")),
    "wrong": [
        (i, DOCLING_RAW["items"][i]["top"][0])
        for i, a, b in zip(main_ids, truth_main, docling_pred, strict=True)
        if a != b
    ],
    "edge": {it["id"]: DOCLING_RAW["items"][it["id"]]["top"][0] for it in EDGE},
    **DOCLING_RAW["stats"],
}
common.save_json("G_summary.json", DOCLING)
print(
    f"3-way accuracy {DOCLING['acc_3way']:.3f}, drawing-or-photo only {DOCLING['acc_2way']:.3f}, "
    f"{DOCLING['ms_median']:.1f} ms per image"
)
print("confusion:", DOCLING["confusion"])
print("top class of the mistakes:", DOCLING["wrong"])
print("edge items:", DOCLING["edge"])

3-way accuracy 0.750, drawing-or-photo only 0.961, 27.4 ms per image
confusion: {'drawing': {'drawing': 82, 'photo': 0, 'other': 34}, 'photo': {'drawing': 2, 'photo': 71, 'other': 15}}
top class of the mistakes: [('d_ctc01_pdf', ['geographical_map', 0.7924900054931641]), ('d_ctc01_p1_r72', ['geographical_map', 0.8236418962478638]), ('d_ctc01_p1_jpeg', ['geographical_map', 0.8062636256217957]), ('d_ctc01_p1_photo', ['geographical_map', 0.7539371252059937]), ('d_ctc02_p1_rotate', ['engineering_drawing', 0.3990836441516876]), ('d_ctc04_p1_photo', ['geographical_map', 0.48529481887817383]), ('d_ctc05_pdf', ['engineering_drawing', 0.49841195344924927]), ('d_ctc05_p1_photo', ['engineering_drawing', 0.230507031083107]), ('d_ctc05_p2_r150', ['engineering_drawing', 0.4125361442565918]), ('d_ctc05_p2_rotate', ['screenshot_from_computer', 0.4519657790660858]), ('d_ftc07_p3_rotate', ['screenshot_from_computer', 0.32689860463142395]), ('d_ftc07_p4_crop', ['engineering_drawing', 0.4313824474811554])

The Docling features as a fourth probe, with the same checks as section B.

In [13]:
saved = np.load(docling_features_path, allow_pickle=True)
EMB["docling_fig"] = dict(zip(saved["ids"], saved["emb"], strict=True))
EMB_STATS["docling_fig"] = json.loads(str(saved["stats"]))
ENCODERS.append("docling_fig")
PROBES["docling_fig"] = run_probe("docling_fig")
common.save_json("B_probe.json", PROBES)
probe = PROBES["docling_fig"]
shown = (
    "C",
    "random_half",
    "grouped_half",
    "lodo_mean",
    "lodo_min",
    "loro",
    "ood_commons",
    "oof_acc",
    "brier",
    "ece",
)
print({k: probe[k] for k in shown})
print("OOF wrong:", probe["oof_wrong"], "OOD wrong:", probe["ood_wrong"])

{'C': 100.0, 'random_half': [0.9901960784313726, 1.0], 'grouped_half': [1.0, 0.9895833333333334], 'lodo_mean': 1.0, 'lodo_min': 1.0, 'loro': {np.str_('ntsb_16-006'): 1.0, np.str_('ntsb_18-061'): 1.0, np.str_('ntsb_24-013'): 1.0}, 'ood_commons': 0.9473684210526315, 'oof_acc': 0.9950980392156863, 'brier': 0.004924771209433145, 'ece': 0.009052018196134855}
OOF wrong: ['d_commons_20200320165124autodeskfu'] OOD wrong: ['d_commons_20200320165124autodeskfu']


## D and F. A local model asked one question

`run_models.py` runs each model in its own process (so peak memory is its own) over every
attachment, resized to 768 px on the long side, with three options: drawing, photo, or other
(a page of text, a logo, a chart). For the 10 messages sent with an attachment it asks twice:
once with the image alone and once with the user's message in the prompt, to see whether a
misleading message sways the answer.

- **D**, general vision-language models: Qwen3-VL 2B, 4B and 8B Instruct, MLX 4-bit
  (Apache-2.0). The question lists the options as letters A, B and C; the probability of each
  option is the softmax of the first answer token's log-probabilities over those three letters.
- **F**, open-weight decision models: LiquidAI d1-omni-600M (8-bit), d1-2B and d1-3B (4-bit),
  licence LFM Open License v1.0. They are built for typed choices: mlx-vlm's `decision.predict`
  takes the named options with a description each and returns a probability per option
  without generating text. This is the local counterpart of hosted decision APIs such as
  Typesafe's Jev or OpenAI's Decisions API.

Four files were added to the set after these models ran (the drawing package, the OCRed
drawing, and the scanned report page with and without OCR), and the models were not run again:
each needs a 2 to 7 GB load. Each model is scored on the files it answered (`scored`), and
`not_run` lists the rest.

In [14]:
VLMS = {
    "qwen3vl_2b": "D",
    "qwen3vl_4b": "D",
    "qwen3vl_8b": "D",
    "d1_omni_600m": "F",
    "d1_2b": "F",
    "d1_3b": "F",
}
MODEL_RES = {}
for name in VLMS:
    if common.load_json(f"model_{name}.json") is None:  # about ten minutes per model
        subprocess.run([sys.executable, str(HERE / "run_models.py"), name], check=True)
    MODEL_RES[name] = common.load_json(f"model_{name}.json")


def model_ece(confidence, correct, bins=5):
    """Expected calibration error over three options, so the bins start at about 1/3."""
    confidence, correct = np.asarray(confidence), np.asarray(correct, float)
    edges = np.linspace(min(confidence.min(), 0.34), 1.0, bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:], strict=True):
        in_bin = (confidence >= lo) & ((confidence < hi) if hi < 1 else (confidence <= hi))
        if in_bin.any():
            ece += in_bin.mean() * abs(correct[in_bin].mean() - confidence[in_bin].mean())
    return float(ece)


def model_scores(name):
    answers = MODEL_RES[name]["items"]
    ids = [i for i in main_ids if i in answers]
    truth = [BY_ID[i]["label"] for i in ids]
    pred = [answers[i]["pred"] for i in ids]
    confidence = [max(answers[i]["p"].values()) for i in ids]
    right = [a == b for a, b in zip(truth, pred, strict=True)]
    pair_truth = [it["label"] for it in PAIRS]
    with_text = [MODEL_RES[name]["pairs"][it["id"]]["pred"] for it in PAIRS]
    image_only = [answers[it["attachment"]]["pred"] for it in PAIRS]
    wrong_confidence = [c for c, ok in zip(confidence, right, strict=True) if not ok]
    return {
        "approach": VLMS[name],
        "acc": float(np.mean(right)),
        "confusion": common.confusion(truth, pred, rows=("drawing", "photo")),
        "scored": len(ids),
        "not_run": [it["id"] for it in ATTACH if it["id"] not in answers],
        "wrong": [
            (i, answers[i]["pred"], round(max(answers[i]["p"].values()), 3))
            for i, ok in zip(ids, right, strict=True)
            if not ok
        ],
        "ece": model_ece(confidence, right),
        "mean_conf_when_wrong": float(np.mean(wrong_confidence)) if wrong_confidence else None,
        "pairs_acc_image_only": accuracy(pair_truth, image_only),
        "pairs_acc_with_text": accuracy(pair_truth, with_text),
        "pairs_flipped_by_text": [
            it["id"] for it, a, b in zip(PAIRS, image_only, with_text, strict=True) if a != b
        ],
        "edge": {it["id"]: answers[it["id"]]["pred"] for it in EDGE if it["id"] in answers},
        **MODEL_RES[name]["stats"],
    }


MODEL_SCORES = {name: model_scores(name) for name in VLMS}
common.save_json("DF_models.json", MODEL_SCORES)
lists = ("confusion", "wrong", "edge", "pairs_flipped_by_text", "not_run")
pd.DataFrame(
    {
        name: {k: v for k, v in scores.items() if k not in lists}
        for name, scores in MODEL_SCORES.items()
    }
).T.round(3)

,approach,acc,scored,ece,mean_conf_when_wrong,pairs_acc_image_only,pairs_acc_with_text,load_s,ms_median,ms_p95,peak_mlx_gb,rss_delta_mb
qwen3vl_2b,D,1.0,202,0.000305,None,1.0,1.0,4.605942,1095.238979,1231.670817,2.948251,56.016896
qwen3vl_4b,D,0.99505,202,0.00448,0.731058,1.0,1.0,5.327362,2009.315438,2256.480729,4.262534,863.125504
qwen3vl_8b,D,0.99505,202,0.004737,0.962619,1.0,1.0,6.477237,3451.752146,3866.67544,6.882054,253.1328
d1_omni_600m,F,0.925743,202,0.0705,0.6673,0.9,0.6,3.764327,132.397542,139.643204,1.655829,592.77312
d1_2b,F,0.99505,202,0.049489,0.478094,1.0,1.0,4.780207,655.197709,4237.859423,2.392306,-141.459456
d1_3b,F,0.985149,202,0.077367,0.553133,1.0,1.0,4.594967,753.134854,5222.403147,3.182842,-169.623552


In [15]:
for name, scores in MODEL_SCORES.items():
    print(f"{name} ({scores['approach']}): confusion {scores['confusion']}")
    print("   wrong:", scores["wrong"])
    print("   flipped by the user's text:", scores["pairs_flipped_by_text"])
    print("   edge items:", scores["edge"])

qwen3vl_2b (D): confusion {'drawing': {'drawing': 114, 'photo': 0}, 'photo': {'drawing': 0, 'photo': 88}}
   wrong: []
   flipped by the user's text: []
   edge items: {'o_logo_18061_x677': 'other', 'o_logo_24013_x159': 'other', 'o_report_16006_pdf': 'other', 'o_report_18061_pdf': 'other', 'o_report_24013_pdf': 'other', 'o_report_16006_page2': 'other', 'o_report_18061_page3': 'photo', 'o_commons_abmessungen': 'drawing'}
qwen3vl_4b (D): confusion {'drawing': {'drawing': 113, 'photo': 0, 'other': 1}, 'photo': {'drawing': 0, 'photo': 88, 'other': 0}}
   wrong: [('d_commons_abgewickeltesblech01', 'other', 0.731)]
   flipped by the user's text: []
   edge items: {'o_logo_18061_x677': 'other', 'o_logo_24013_x159': 'other', 'o_report_16006_pdf': 'other', 'o_report_18061_pdf': 'other', 'o_report_24013_pdf': 'other', 'o_report_16006_page2': 'other', 'o_report_18061_page3': 'other', 'o_commons_abmessungen': 'drawing'}
qwen3vl_8b (D): confusion {'drawing': {'drawing': 113, 'photo': 0, 'other': 1}

## E. The layered router

Layer 1 is the structure-only rules (no attachment, vector PDF or drawing package, prose PDF).
What they leave undecided goes to layer 2, a linear probe whose probability comes from the
out-of-fold predictions above, so no item is scored by a probe that saw its drawing, its report
or its Commons file. When the probe's confidence max(p, 1 - p) is below the threshold t, layer 3
asks a local model and takes its answer, which may also be "other". The user's text is not
shown to the router; the effect of showing it is measured separately above.

Every file is counted once: a message with an attachment repeats a file that is already in the
set, and the router ignores the message, so the pairs are checked to route like their files and
then left out of the counts. Scores are over the 224 labelled requests (204 drawing and photo
files and 20 questions); the 10 files that fit no case are reported apart.

The times are reconstructed, not measured end to end: each request gets the measured cost of
every layer it passed through. That is the layer-1 check of that PDF (section A), or a 1 ms
allowance for an image's file-type check, then the embedding median and the probe time
(section B), then that file's own model call when it escalates.

The escalation rate depends on how the probe's folds are grouped, so every summary is computed
three ways: 5 folds grouped by source (the default), 5 folds grouped by report page, and
leaving one source out at a time.

In [16]:
LAYER1 = {
    "structural": dict(zip(rules_structure.id, rules_structure.decision, strict=True)),
    "full": dict(zip(rules_exif.id, rules_exif.decision, strict=True)),
    # no layer 1 beyond "no attachment is text": every attachment goes to the probe
    "none": {it["id"]: "text" if it["kind"] == "text" else "undecided" for it in ITEMS},
}
GROUPINGS = ("report", "page", "loso")  # report: 5 folds by source, the default
PROBE_P = {
    grouping: {
        name: {
            **(
                PROBES[name]["oof_p_drawing"]
                if grouping == "report"
                else PROBES[name]["oof_by_grouping"][grouping]
            ),
            **PROBES[name]["edge"],
        }
        for name in ENCODERS
    }
    for grouping in GROUPINGS
}


def file_of(it):
    """The item whose file a request sends."""
    return it["attachment"] if it["variant"] == "text_pair" else it["id"]


def layer1_ms(it, rules):
    features = FEATS[it["id"]]
    if features["kind"] == "text":
        return 0.0
    if rules == "none":
        return 1.0  # a file-type check only; a PDF page is rendered inside the embedding time
    if rules == "structural":
        # a file-type check for an image; the measured layer-1 check of that PDF
        return 1.0 if features["kind"] == "image" else STRUCT10_MS[file_of(it)]
    return features["ms"]


def layered(encoder, model, threshold, rules="structural", grouping="report"):
    """Every item routed through the layers: its final answer, the stage that gave it and the
    reconstructed time. A threshold above 1 skips the probe, so every file goes to the model."""
    rows = []
    for it in ITEMS:
        decision = LAYER1[rules][it["id"]]
        ms = layer1_ms(it, rules)
        stage = "A"
        if decision == "undecided":
            stage = "B"
            if threshold <= 1:
                p_drawing = PROBE_P[grouping][encoder][file_of(it)]
                ms += EMB_STATS[encoder]["ms_median"] + PROBES[encoder]["predict_ms"]
                if max(p_drawing, 1 - p_drawing) >= threshold:
                    decision = label_of(p_drawing)
            if decision == "undecided":
                stage = "D"
                answer = MODEL_RES[model]["items"].get(file_of(it))
                if answer is None:  # a file the model was not run on
                    decision = "not run"
                else:
                    decision = answer["pred"]
                    ms += answer["ms"]
        rows.append(
            {
                "id": it["id"],
                "label": it["label"],
                "variant": it["variant"],
                "final": decision,
                "stage": stage,
                "ms": ms,
            }
        )
    return pd.DataFrame(rows)


def layered_summary(df):
    final = dict(zip(df.id, df.final, strict=True))
    for it in PAIRS:  # the router ignores the message, so a pair routes like its file
        assert final[it["id"]] == final[it["attachment"]], it["id"]
    df = df[df.variant != "text_pair"]  # count every file once
    labelled = df[df.label != "other"]
    files = labelled[labelled.variant != "text_only"]  # the unique drawing and photo files
    edge = df[df.label == "other"]
    measured = labelled[labelled.final != "not run"]
    return {
        "labelled": len(labelled),
        "attachments": len(files),
        "acc_all": float((measured.final == measured.label).mean()),
        "acc_attachments": float((files.final == files.label).mean()),
        "errors": int((measured.final != measured.label).sum()),
        "not_run": int((df.final == "not run").sum()),
        "not_run_labelled": int((labelled.final == "not run").sum()),
        "not_run_edge": int((edge.final == "not run").sum()),
        "share_A": float((files.stage == "A").mean()),
        "share_B": float((files.stage == "B").mean()),
        "escalated": float((files.stage == "D").mean()),
        "escalated_n": int((files.stage == "D").sum()),
        "ms_mean_attachment": float(files.ms.mean()),
        "ms_median_attachment": float(files.ms.median()),
        "ms_p95_attachment": float(files.ms.quantile(0.95)),
        "edge": len(edge),
        "edge_not_forced": int(edge.final.isin(["other", "document"]).sum()),
    }


THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.98, 0.99, 0.995]
sweep = []
for grouping in GROUPINGS:
    for encoder in ENCODERS:
        for model in VLMS:
            for threshold in THRESHOLDS:
                summary = layered_summary(layered(encoder, model, threshold, grouping=grouping))
                sweep.append(
                    {"grouping": grouping, "encoder": encoder, "model": model, "t": threshold}
                    | summary
                )
SWEEP = pd.DataFrame(sweep)
common.save_json("E_sweep.json", SWEEP.to_dict("records"))
REC_SWEEP = SWEEP[(SWEEP.encoder == "dinov2_s") & (SWEEP.model == "qwen3vl_8b")]
REC_SWEEP.pivot(index="t", columns="grouping", values="escalated").round(3)

grouping,loso,page,report
t,,,
0.500,0.000,0.000,0.000
0.600,0.000,0.000,0.000
0.700,0.000,0.000,0.000
0.800,0.020,0.005,0.015
0.900,0.039,0.025,0.029
0.950,0.108,0.064,0.108
0.980,0.186,0.118,0.206
0.990,0.289,0.186,0.299
0.995,0.446,0.348,0.461


In [17]:
DEFAULT_GROUPING = SWEEP[SWEEP.grouping == "report"]
print("The recommended router at each threshold (5 folds grouped by source):")
display(
    REC_SWEEP[REC_SWEEP.grouping == "report"][
        ["t", "errors", "escalated", "ms_mean_attachment", "edge_not_forced"]
    ].round(3)
)
print("Every encoder with Qwen3-VL 8B as layer 3, at t = 0.9, under each grouping:")
display(
    SWEEP[(SWEEP.model == "qwen3vl_8b") & (SWEEP.t == 0.9)]
    .pivot(index="encoder", columns="grouping", values="escalated")
    .round(3)
)
print("Every layer-3 model behind DINOv2 at t = 0.9:")
display(
    DEFAULT_GROUPING[(DEFAULT_GROUPING.encoder == "dinov2_s") & (DEFAULT_GROUPING.t == 0.9)].round(
        3
    )
)

The recommended router at each threshold (5 folds grouped by source):


,t,errors,escalated,ms_mean_attachment,edge_not_forced
126,0.500,0,0.000,33.046,3
127,0.600,0,0.000,33.046,4
128,0.700,0,0.000,33.046,5
129,0.800,0,0.015,74.264,5
130,0.900,0,0.029,111.339,6
131,0.950,0,0.108,356.173,7
132,0.980,0,0.206,678.874,7
133,0.990,0,0.299,980.669,7
134,0.995,0,0.461,1504.104,7


Every encoder with Qwen3-VL 8B as layer 3, at t = 0.9, under each grouping:


grouping,loso,page,report
encoder,,,
clip_b32,0.054,0.029,0.054
dinov2_s,0.039,0.025,0.029
docling_fig,0.039,0.015,0.044
siglip2_b16,0.049,0.025,0.054


Every layer-3 model behind DINOv2 at t = 0.9:


,grouping,encoder,model,t,labelled,attachments,acc_all,acc_attachments,errors,not_run,not_run_labelled,not_run_edge,share_A,share_B,escalated,escalated_n,ms_mean_attachment,ms_median_attachment,ms_p95_attachment,edge,edge_not_forced
112,report,dinov2_s,qwen3vl_2b,0.9,224,204,1.000,1.00,0,2,0,2,0.059,0.912,0.029,6,57.139,33.507,33.691,10,6
121,report,dinov2_s,qwen3vl_4b,0.9,224,204,1.000,1.00,0,2,0,2,0.059,0.912,0.029,6,77.970,33.507,33.691,10,6
130,report,dinov2_s,qwen3vl_8b,0.9,224,204,1.000,1.00,0,2,0,2,0.059,0.912,0.029,6,111.339,33.507,33.691,10,6
139,report,dinov2_s,d1_omni_600m,0.9,224,204,0.991,0.99,2,2,0,2,0.059,0.912,0.029,6,36.806,33.507,33.691,10,5
148,report,dinov2_s,d1_2b,0.9,224,204,1.000,1.00,0,2,0,2,0.059,0.912,0.029,6,102.759,33.507,33.691,10,6
157,report,dinov2_s,d1_3b,0.9,224,204,1.000,1.00,0,2,0,2,0.059,0.912,0.029,6,61.102,33.507,33.691,10,6


For contrast, the same router with the full pixel rules (EXIF, white and flat shares) as
layer 1: every photographed drawing is decided wrongly before the classifier sees it.

In [18]:
pd.DataFrame(
    [
        {
            "encoder": "dinov2_s",
            "model": model,
            "t": 0.9,
            **layered_summary(layered("dinov2_s", model, 0.9, rules="full")),
        }
        for model in ("d1_omni_600m", "qwen3vl_8b")
    ]
).round(3)

,encoder,model,t,labelled,attachments,acc_all,acc_attachments,errors,not_run,not_run_labelled,not_run_edge,share_A,share_B,escalated,escalated_n,ms_mean_attachment,ms_median_attachment,ms_p95_attachment,edge,edge_not_forced
0,dinov2_s,d1_omni_600m,0.9,224,204,0.933,0.926,15,2,0,2,0.647,0.328,0.025,5,61.201,52.806,130.828,10,4
1,dinov2_s,qwen3vl_8b,0.9,224,204,0.938,0.931,14,2,0,2,0.647,0.328,0.025,5,119.637,52.806,130.828,10,5


**Why not one model alone.** The DINOv2 probe made no mistake on the drawings and photos, so
the simplest router is the probe plus "no attachment is text". At a threshold of 0.5 nothing
escalates, so these rows compare the probe alone, the probe behind the structure rules, the
probe with the third layer, and the recommended router with both. The decision model d1-2B can
answer "other" itself, so it is also tried alone and behind the structure rules, with no probe
(a threshold above 1 sends every file to the model).

In [19]:
simpler = []
for router, rules, threshold, model in [
    ("probe alone", "none", 0.5, "qwen3vl_8b"),
    ("structure rules and probe", "structural", 0.5, "qwen3vl_8b"),
    ("probe and Qwen3-VL 8B below 0.9", "none", 0.9, "qwen3vl_8b"),
    ("recommended", "structural", 0.9, "qwen3vl_8b"),
    ("d1-2B alone", "none", 1.01, "d1_2b"),
    ("structure rules and d1-2B", "structural", 1.01, "d1_2b"),
]:
    df = layered("dinov2_s", model, threshold, rules=rules)
    edge = df[df.label == "other"]
    simpler.append(
        {
            "router": router,
            "rules": rules,
            "t": threshold,
            "model": model,
            **layered_summary(df),
            "edge_final": dict(zip(edge.id, edge.final, strict=True)),
        }
    )
display(pd.DataFrame(simpler).drop(columns="edge_final").round(3))
common.save_json("E_simpler.json", simpler)

,router,rules,t,model,labelled,attachments,acc_all,acc_attachments,errors,not_run,not_run_labelled,not_run_edge,share_A,share_B,escalated,escalated_n,ms_mean_attachment,ms_median_attachment,ms_p95_attachment,edge,edge_not_forced
0,probe alone,none,0.50,qwen3vl_8b,224,204,1.000,1.000,0,0,0,0,0.000,1.000,0.000,0,33.507,33.507,33.507,10,0
1,structure rules and probe,structural,0.50,qwen3vl_8b,224,204,1.000,1.000,0,0,0,0,0.059,0.941,0.000,0,33.046,33.507,33.617,10,3
2,probe and Qwen3-VL 8B below 0.9,none,0.90,qwen3vl_8b,224,204,1.000,0.995,0,3,1,2,0.000,0.966,0.034,7,111.800,33.507,33.507,10,3
3,recommended,structural,0.90,qwen3vl_8b,224,204,1.000,1.000,0,2,0,2,0.059,0.912,0.029,6,111.339,33.507,33.691,10,6
4,d1-2B alone,none,1.01,d1_2b,224,204,0.995,0.985,1,4,2,2,0.000,0.000,1.000,204,1179.911,656.198,4291.712,10,7
5,structure rules and d1-2B,structural,1.01,d1_2b,224,204,0.996,0.990,1,3,1,2,0.059,0.000,0.941,192,1146.466,649.592,4291.712,10,7


## Errors of the recommended configuration

In [20]:
REC = {"encoder": "dinov2_s", "model": "qwen3vl_8b", "t": 0.9}
rec_df = layered(REC["encoder"], REC["model"], REC["t"])
print(layered_summary(rec_df))
display(pd.crosstab(rec_df.label, rec_df.final, margins=True))
display(rec_df[rec_df.label != rec_df.final])
display(rec_df[rec_df.stage == "D"])

{'labelled': 224, 'attachments': 204, 'acc_all': 1.0, 'acc_attachments': 1.0, 'errors': 0, 'not_run': 2, 'not_run_labelled': 0, 'not_run_edge': 2, 'share_A': 0.058823529411764705, 'share_B': 0.9117647058823529, 'escalated': 0.029411764705882353, 'escalated_n': 6, 'ms_mean_attachment': 111.33923117222045, 'ms_median_attachment': 33.50655899755657, 'ms_p95_attachment': 33.691379555966705, 'edge': 10, 'edge_not_forced': 6}


final,document,drawing,not run,other,photo,text,All
label,,,,,,,
drawing,0,121,0,0,0,0,121
other,3,2,2,3,0,0,10
photo,0,0,0,0,93,0,93
text,0,0,0,0,0,20,20
All,3,123,2,3,93,20,244


,id,label,variant,final,stage,ms
204,o_report_16006_pdf,other,report_pdf,document,A,30.302584
205,o_report_18061_pdf,other,report_pdf,document,A,36.184500
206,o_report_24013_pdf,other,report_pdf,document,A,6.855708
208,o_report_18061_page3,other,report_page,drawing,B,33.506559
209,o_commons_abmessungen,other,diagram,drawing,B,33.506559
212,o_report_18061_page5_scan,other,report_scan_pdf,not run,D,33.859351
213,o_report_18061_page5_ocrscan,other,report_ocr_scan_pdf,not run,D,36.998476


,id,label,variant,final,stage,ms
109,d_commons_aircompressorverticalsec,drawing,commons,drawing,D,2797.131726
115,p_16006_p06_x33,photo,ntsb_figure,photo,D,2428.218017
118,p_16006_p08_x63,photo,ntsb_figure,photo,D,2438.554226
119,p_16006_p08_x64,photo,ntsb_figure,photo,D,2625.064475
129,p_16006_p13_x130,photo,ntsb_figure,photo,D,2431.808726
180,o_logo_18061_x677,other,logo,other,D,1214.890059
181,o_logo_24013_x159,other,logo,other,D,1200.279601
201,p_commons_ourbrokenvcrpart3detail7,photo,commons,photo,D,3452.041726
207,o_report_16006_page2,other,report_page,other,D,3488.871684
212,o_report_18061_page5_scan,other,report_scan_pdf,not run,D,33.859351


What each layer of the recommended router decided, over the 234 unique requests (the 10 pairs
route like their files and are left out). "Reached" counts the requests the earlier layers
passed on. The time is the reconstructed cost of a request that ends at that layer, earlier
layers included, split by what the request carried.

In [21]:
def input_of(item_id):
    kind = BY_ID[item_id]["kind"]
    return "none" if kind == "text" else ("pdf" if kind.endswith("pdf") else "image")


def layer_table(df):
    df = df[df.variant != "text_pair"]
    rows, reached = [], len(df)
    for stage in ("A", "B", "D"):
        decided = df[df.stage == stage]
        labelled = decided[decided.label != "other"]
        edge = decided[decided.label == "other"]
        rows.append(
            {
                "stage": stage,
                "reached": reached,
                "decided": len(decided),
                "decided_as": dict(Counter(decided.final)),
                "labelled": len(labelled),
                "wrong": int((labelled.final != labelled.label).sum()),
                "edge": len(edge),
                "edge_forced": int(edge.final.isin(["drawing", "photo"]).sum()),
                "ms_median": float(decided.ms.median()),
                "ms_min": float(decided.ms.min()),
                "ms_max": float(decided.ms.max()),
                "ms_median_by_input": {
                    k: float(v)
                    for k, v in decided.groupby(decided.id.map(input_of)).ms.median().items()
                },
                "ids": list(decided.id) if stage == "D" else [],
            }
        )
        reached -= len(decided)
    return rows


rec_layers = layer_table(rec_df)
display(pd.DataFrame(rec_layers).drop(columns="ids"))
unique_df = rec_df[rec_df.variant != "text_pair"]
rec_files = unique_df[unique_df.id.map(input_of) != "none"]
rec_totals = {
    "requests": len(unique_df),
    "attachments": len(rec_files),
    "attachments_escalated": int((rec_files.stage == "D").sum()),
    "ms_mean_all_attachments": float(rec_files.ms.mean()),
}
print(rec_totals)

,stage,reached,decided,decided_as,labelled,wrong,edge,edge_forced,ms_median,ms_min,ms_max,ms_median_by_input
0,A,234,35,"{'drawing': 12, 'document': 3, 'text': 20}",32,0,3,0,0.000000,0.000000,50.492458,"{'none': 0.0, 'pdf': 22.91579102165997}"
1,B,199,188,"{'drawing': 105, 'photo': 83}",186,0,2,2,33.506559,33.506559,36.033267,"{'image': 33.50655899755657, 'pdf': 33.63203799817711}"
2,D,11,11,"{'drawing': 1, 'photo': 5, 'other': 3, 'not run': 2}",6,0,5,0,2431.808726,33.859351,3488.871684,"{'image': 2438.55422597751, 'pdf': 35.42891349643469}"


{'requests': 234, 'attachments': 214, 'attachments_escalated': 11, 'ms_mean_all_attachments': 134.71242168339438}


**How much the escalation rate depends on the grouping**, and what that means at a million
files: the escalated share times a million, times Qwen3-VL 8B's median time per image over
all items, and the probe's embedding time for every file.

In [22]:
SCALE = 1_000_000
qwen_s = MODEL_SCORES[REC["model"]]["ms_median"] / 1000
rec_range = {}
for grouping in GROUPINGS:
    df = layered(REC["encoder"], REC["model"], REC["t"], grouping=grouping)
    summary = layered_summary(df)
    rec_range[grouping] = summary | {
        "escalated_ids": list(
            df.query("stage == 'D' and label in ('drawing', 'photo') and variant != 'text_pair'").id
        ),
        "million_escalated": summary["escalated"] * SCALE,
        "million_layer3_hours": summary["escalated"] * SCALE * qwen_s / 3600,
    }
rec_scale = {
    "probe_hours_per_million": SCALE * EMB_STATS[REC["encoder"]]["ms_median"] / 1000 / 3600,
    "qwen_s_per_image": qwen_s,
}
shown = [
    "escalated_n",
    "escalated",
    "ms_mean_attachment",
    "million_layer3_hours",
    "edge_not_forced",
]
display(pd.DataFrame(rec_range).T[shown])
print(rec_scale)
common.save_json(
    "E_recommended.json",
    {
        "config": REC,
        "summary": layered_summary(rec_df),
        "by_grouping": rec_range,
        "scale": rec_scale,
        "layers": rec_layers,
        "totals": rec_totals,
        "confusion": pd.crosstab(rec_df.label, rec_df.final).to_dict(),
        "rows": rec_df.to_dict("records"),
    },
)

,escalated_n,escalated,ms_mean_attachment,million_layer3_hours,edge_not_forced
report,6,0.029412,111.339231,28.200589,6
page,5,0.02451,103.790818,23.500491,6
loso,8,0.039216,140.657367,37.600786,6


{'probe_hours_per_million': 9.018113552075294, 'qwen_s_per_image': 3.451752145541832}


## Summary table

One row per approach, as used in the write-up. Accuracy is on the 204 drawing and photo files
(messages with an attachment counted once, through their file); a decision of "undecided" or
"other" counts as wrong. Precision for a case is the share of the files an option sent to that
case that belong there, and recall is the share of that case's files it sent there, so an
undecided or "other" answer lowers recall but not precision. Latency is the median per file on
the M1 Pro.

In [23]:
def precision_recall(truth, pred):
    """Precision and recall of the drawing and photo cases."""
    out = {}
    for case in ("drawing", "photo"):
        called = [t for t, q in zip(truth, pred, strict=True) if q == case]
        actual = [q for t, q in zip(truth, pred, strict=True) if t == case]
        out[f"{case}_precision"] = sum(t == case for t in called) / len(called) if called else None
        out[f"{case}_recall"] = sum(q == case for q in actual) / len(actual)
    return out


def rules_row(name, decisions, ms):
    df = decisions[decisions.id.isin(main_ids)]
    decided = df[df.decision.isin(["drawing", "photo"])]
    return {
        "approach": name,
        "accuracy": float((df.decision == df.label).mean()),
        "coverage": float(len(decided) / len(df)),
        "precision_when_decided": float((decided.decision == decided.label).mean()),
        **precision_recall(list(df.label), list(df.decision)),
        "ms_median": ms,
        "memory_mb": None,
        "probability": "no",
    }


cad_pdf_ms = float(np.median([STRUCT10_MS[it["id"]] for it in pdf_items if it["variant"] == "pdf"]))
rows = [
    rules_row("A rules with EXIF", rules_exif, float(features_df.ms.median())),
    rules_row("A rules without EXIF", rules_no_exif, float(features_df.ms.median())),
    # the layer-1 check on the CAD PDFs it decides (an image only needs its file type)
    rules_row("A structure only", rules_structure, cad_pdf_ms),
]
for name in ENCODERS:
    oof_pred = [label_of(PROBES[name]["oof_p_drawing"][i]) for i in main_ids]
    rows.append(
        {
            "approach": f"B probe on {name}",
            "accuracy": PROBES[name]["oof_acc"],
            **precision_recall(truth_main, oof_pred),
            "ood_commons": PROBES[name]["ood_commons"],
            "lodo_mean": PROBES[name]["lodo_mean"],
            "coverage": 1.0,
            "ms_median": EMB_STATS[name]["ms_median"],
            "memory_mb": EMB_STATS[name]["mps_driver_mb"],
            "probability": f"yes (ECE {PROBES[name]['ece']:.3f})",
        }
    )
for name in ("clip_b32", "siglip2_b16"):
    pred = [label_of(ZERO_SHOT[name]["p_drawing"][i]) for i in main_ids]
    rows.append(
        {
            "approach": f"C zero-shot {name}",
            "accuracy": ZERO_SHOT[name]["acc"],
            **precision_recall(truth_main, pred),
            "coverage": 1.0,
            "ms_median": EMB_STATS[name]["ms_median"],
            "memory_mb": EMB_STATS[name]["mps_driver_mb"],
            "probability": "softmax or sigmoid score",
        }
    )
rows.append(
    {
        "approach": "G Docling figure classifier",
        "accuracy": DOCLING["acc_3way"],
        **precision_recall(truth_main, docling_pred),
        "coverage": 1.0,
        "ms_median": DOCLING["ms_median"],
        "memory_mb": DOCLING["mps_driver_mb"],
        "probability": "softmax over 26 classes",
    }
)
for name, scores in MODEL_SCORES.items():
    answers = MODEL_RES[name]["items"]
    ids = [i for i in main_ids if i in answers]
    rows.append(
        {
            "approach": f"{scores['approach']} {name}",
            "accuracy": scores["acc"],
            "scored": scores["scored"],
            **precision_recall([BY_ID[i]["label"] for i in ids], [answers[i]["pred"] for i in ids]),
            "coverage": 1.0,
            "ms_median": scores["ms_median"],
            "memory_mb": scores["peak_mlx_gb"] * 1000,
            "probability": f"yes (ECE {scores['ece']:.3f})",
        }
    )
rec_summary = layered_summary(rec_df)
rec_main = rec_df[rec_df.id.isin(main_ids)]
rows.append(
    {
        "approach": "E layered (structure, DINOv2 probe, Qwen3-VL 8B at t=0.9)",
        "accuracy": rec_summary["acc_attachments"],
        **precision_recall(list(rec_main.label), list(rec_main.final)),
        "coverage": 1.0,
        "ms_median": rec_summary["ms_median_attachment"],
        "escalated": rec_summary["escalated"],
        "probability": "yes",
    }
)
SUMMARY = pd.DataFrame(rows)
common.save_json("summary.json", SUMMARY.to_dict("records"))
SUMMARY.round(3)

,approach,accuracy,coverage,precision_when_decided,drawing_precision,drawing_recall,photo_precision,photo_recall,ms_median,memory_mb,probability,ood_commons,lodo_mean,scored,escalated
0,A rules with EXIF,0.578,0.647,0.894,1.000,0.767,0.674,0.330,38.042,NaN,no,NaN,NaN,NaN,NaN
1,A rules without EXIF,0.294,0.358,0.822,1.000,0.328,0.629,0.250,38.042,NaN,no,NaN,NaN,NaN,NaN
2,A structure only,0.059,0.059,1.000,1.000,0.103,NaN,0.000,21.171,NaN,no,NaN,NaN,NaN,NaN
3,B probe on clip_b32,1.000,1.000,NaN,1.000,1.000,1.000,1.000,30.559,1615.479,yes (ECE 0.023),0.895,1.0,NaN,NaN
4,B probe on siglip2_b16,1.000,1.000,NaN,1.000,1.000,1.000,1.000,39.478,1548.370,yes (ECE 0.021),0.895,1.0,NaN,NaN
5,B probe on dinov2_s,1.000,1.000,NaN,1.000,1.000,1.000,1.000,32.465,113.902,yes (ECE 0.019),1.000,1.0,NaN,NaN
6,B probe on docling_fig,0.995,1.000,NaN,1.000,0.991,0.989,1.000,27.400,72.155,yes (ECE 0.009),0.947,1.0,NaN,NaN
7,C zero-shot clip_b32,0.990,1.000,NaN,0.983,1.000,1.000,0.977,30.559,1615.479,softmax or sigmoid score,NaN,NaN,NaN,NaN
8,C zero-shot siglip2_b16,1.000,1.000,NaN,1.000,1.000,1.000,1.000,39.478,1548.370,softmax or sigmoid score,NaN,NaN,NaN,NaN
9,G Docling figure classifier,0.750,1.000,NaN,0.976,0.707,1.000,0.807,27.400,72.155,softmax over 26 classes,NaN,NaN,NaN,NaN
